<!-- @format -->

# Adult Income Prediction

## 3. Classical Machine Learning Pipeline

Notebook này xây dựng đầy đủ classical ML pipeline, bao gồm:

1. Tái lập preprocessing từ notebook 02
2. Trích xuất đặc trưng: Full features vs PCA 95%
3. Huấn luyện baseline: Logistic Regression, Linear SVM, Random Forest, Gaussian Naive Bayes
4. Hyperparameter tuning với cross-validation
5. So sánh và lựa chọn mô hình cuối cùng
6. Phân tích feature importance
7. Lưu features, kết quả và mô hình


In [ ]:
# [Pipeline 3.0] Import modules và tái lập preprocessing
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import modules.preprocessing as prep
import modules.classical_learning as cl

# Load data
url = "https://raw.githubusercontent.com/Hanne2202/ml-group10-data/main/adult.csv"
df = pd.read_csv(url)

# Tái lập preprocessing (giống 02_preprocessing.ipynb)
df_prep = df.copy()
df_prep = prep.drop_columns(df_prep, ['education', 'fnlwgt'])
df_prep.replace('?', np.nan, inplace=True)

X = df_prep.drop(columns=['income'])
y = df_prep['income']

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Best preprocessor từ 02_preprocessing.ipynb
best_preprocessor = prep.build_classical_preprocessor(
    'config_2_onehot_constant_standard', num_features, cat_features
)
X_train_processed = best_preprocessor.fit_transform(X_train)
X_test_processed  = best_preprocessor.transform(X_test)

print('X_train_processed shape:', X_train_processed.shape)
print('X_test_processed shape :', X_test_processed.shape)

<!-- @format -->

## [Pipeline 3.1] Feature Extraction

Sau bước preprocessing, dữ liệu đã được chuẩn hóa và mã hóa one-hot, tạo ra 91 features.

Trong bước này, nhóm xây dựng hai nhánh đặc trưng để so sánh:

- **Full features**: giữ nguyên toàn bộ 91 features sau preprocessing.
- **PCA 95%**: áp dụng PCA trên tập train và giữ lại số chiều đủ giải thích 95% phương sai.

Lưu ý: PCA chỉ được fit trên tập train, sau đó transform tập test để tránh data leakage.


In [ ]:
# [Pipeline 3.1] Trích xuất đặc trưng: Full features và PCA 95%
feature_sets, y_train_model, y_test_model = cl.extract_features_pca(
    X_train_processed, X_test_processed,
    y_train, y_test,
    pca_variance=0.95
)

for name, data in feature_sets.items():
    print(f"{name}: X_train={data['X_train'].shape}, X_test={data['X_test'].shape}")

In [ ]:
# [Pipeline 3.1] Visualize cumulative explained variance của PCA
cl.plot_pca_variance(feature_sets, pca_key='pca_95')

In [ ]:
# [Pipeline 3.1] Lưu feature files để tái sử dụng
cl.save_features(feature_sets, y_train_model, y_test_model, output_dir='../features')

<!-- @format -->

## [Pipeline 3.2] Baseline Model Training

Sau khi hoàn thành bước trích xuất đặc trưng, nhóm tiến hành huấn luyện các mô hình baseline trên cả hai nhánh đặc trưng:

- `full_features`: giữ nguyên toàn bộ 91 đặc trưng sau preprocessing.
- `pca_95`: giảm chiều bằng PCA và giữ lại 95% phương sai.

Bốn mô hình học máy truyền thống được sử dụng:

1. **Logistic Regression** — baseline tuyến tính mạnh, huấn luyện nhanh.
2. **Linear SVM** — SVM với kernel tuyến tính, hiệu quả tính toán tốt hơn RBF trên tập lớn.
3. **Random Forest** — ensemble phi tuyến, mô hình hóa tương tác giữa các đặc trưng.
4. **Gaussian Naive Bayes** — baseline xác suất, liên hệ với lý thuyết Naive Bayes.

`class_weight='balanced'` được sử dụng để xử lý mất cân bằng lớp. F1-score là chỉ số chính để so sánh.


In [ ]:
# [Pipeline 3.2] Định nghĩa các mô hình baseline
baseline_models = cl.get_baseline_models()

print('Baseline models:')
for name in baseline_models:
    print(' -', name)

In [ ]:
# [Pipeline 3.2] Huấn luyện và đánh giá baseline trên cả hai nhánh đặc trưng
trained_models, baseline_results_df, baseline_predictions = cl.run_baseline(
    baseline_models, feature_sets, y_train_model, y_test_model
)

In [ ]:
# [Pipeline 3.2] Bảng kết quả baseline sắp xếp theo F1-score
baseline_results_df

In [ ]:
# [Pipeline 3.2] Lưu kết quả baseline
cl.save_results(baseline_results_df, 'baseline_results.csv', output_dir='../results')

In [ ]:
# [Pipeline 3.2] Mô hình baseline tốt nhất và biểu đồ so sánh
best_baseline = baseline_results_df.iloc[0]
cl.print_best_model_summary(baseline_results_df, stage='Baseline')

cl.plot_f1_comparison(
    baseline_results_df,
    score_col='f1_score',
    title='Baseline Model Comparison by F1-score'
)

In [ ]:
# [Pipeline 3.2] Confusion matrix và classification report cho mô hình baseline tốt nhất
best_key = f"{best_baseline['feature_set']}__{best_baseline['model']}"
best_y_pred = baseline_predictions[best_key]

cl.plot_confusion_matrix(
    y_test_model, best_y_pred,
    title=f"Confusion Matrix — {best_baseline['model']} ({best_baseline['feature_set']})"
)
cl.print_classification_report(y_test_model, best_y_pred, title=best_key)

<!-- @format -->

### Nhận xét Confusion Matrix

Với mô hình baseline tốt nhất là Logistic Regression trên `full_features`, ma trận nhầm lẫn cho thấy mô hình dự đoán đúng 5924 mẫu thuộc lớp `<=50K` và 1963 mẫu thuộc lớp `>50K`.

Đối với lớp `>50K`, mô hình có recall cao, đạt 0.84, nghĩa là mô hình phát hiện được phần lớn các mẫu có thu nhập cao. Tuy nhiên, precision của lớp này chỉ đạt 0.57, cho thấy vẫn còn khá nhiều mẫu `<=50K` bị dự đoán nhầm thành `>50K`.

Điều này phản ánh đặc điểm mất cân bằng của tập dữ liệu Adult: lớp `<=50K` chiếm số lượng lớn hơn đáng kể so với lớp `>50K`. Vì vậy, F1-score được sử dụng làm chỉ số chính để cân bằng giữa precision và recall, thay vì chỉ dựa vào accuracy.


<!-- @format -->

### Nhận xét kết quả baseline

Trong bước huấn luyện baseline, nhóm so sánh bốn mô hình học máy truyền thống trên hai nhánh đặc trưng:

- `full_features`: giữ nguyên 91 đặc trưng sau bước tiền xử lý.
- `pca_95`: sử dụng PCA để giảm chiều, giữ lại 95% phương sai và thu được 25 thành phần chính.

Dựa trên F1-score, mô hình tốt nhất ở giai đoạn baseline là **Logistic Regression trên tập `full_features`**.

Random Forest đạt accuracy cao nhất nhưng F1-score thấp hơn Logistic Regression, cho thấy accuracy không nên là chỉ số duy nhất khi dữ liệu có mất cân bằng lớp.

Gaussian Naive Bayes có hiệu năng thấp nhất trên `full_features` do vi phạm giả định phân phối Gaussian với dữ liệu one-hot encoding. Sau PCA, GNB cải thiện rõ rệt vì các thành phần chính dạng liên tục phù hợp hơn với giả định của mô hình.

PCA không cải thiện hiệu năng của các mô hình chính. Vì vậy, `full_features` được chọn làm nhánh đặc trưng chính cho bước tuning tiếp theo.


<!-- @format -->

## [Pipeline 3.3] Hyperparameter Tuning

Từ kết quả baseline, nhánh `full_features` cho hiệu quả tốt hơn hoặc tương đương `pca_95` ở hầu hết các mô hình. Vì vậy, trong bước tinh chỉnh tham số, nhóm chọn `full_features` làm nhánh đặc trưng chính.

Việc tinh chỉnh tham số được thực hiện bằng cross-validation trên tập train. Tập test chỉ được sử dụng ở bước đánh giá cuối cùng để tránh data leakage.

Nhóm sử dụng **Stratified K-Fold Cross Validation với 5 folds** để giữ tỷ lệ lớp ổn định. Đối với Random Forest, `RandomizedSearchCV` được dùng với `n_iter=15` để cân bằng phạm vi tìm kiếm và thời gian chạy.

F1-score tiếp tục được chọn làm scoring metric chính.


In [ ]:
# [Pipeline 3.3] Chạy hyperparameter tuning
tuning_results_df, tuned_models, tuned_predictions = cl.run_tuning(
    feature_sets,
    y_train_model,
    y_test_model,
    main_feature_set='full_features',
    n_splits=5,
)

In [ ]:
# [Pipeline 3.3] Bảng kết quả tuning
tuning_results_df

In [ ]:
# [Pipeline 3.3] Lưu kết quả tuning
cl.save_results(tuning_results_df, 'tuning_results.csv', output_dir='../results')

In [ ]:
# [Pipeline 3.3] So sánh F1-score sau tuning
best_tuned = tuning_results_df.iloc[0]
cl.print_best_model_summary(tuning_results_df, stage='Tuned')

cl.plot_f1_comparison(
    tuning_results_df.rename(columns={'test_f1_score': 'f1_score'}),
    score_col='f1_score',
    title='Tuned Model Comparison by Test F1-score'
)

In [ ]:
# [Pipeline 3.3] Confusion matrix và classification report cho mô hình tuned tốt nhất
best_tuned_name   = best_tuned['model']
best_tuned_y_pred = tuned_predictions[best_tuned_name]

cl.plot_confusion_matrix(
    y_test_model, best_tuned_y_pred,
    title=f"Confusion Matrix — Tuned {best_tuned_name} (full_features)"
)
cl.print_classification_report(y_test_model, best_tuned_y_pred, title=f'Tuned {best_tuned_name}')

<!-- @format -->

## [Pipeline 3.4] ROC Curve Visualization

Bên cạnh ROC-AUC score, nhóm trực quan hóa đường cong ROC của các mô hình sau tuning trên cùng một biểu đồ. Biểu đồ ROC giúp quan sát trực quan khả năng phân biệt hai lớp của từng mô hình ở nhiều ngưỡng quyết định khác nhau.


In [ ]:
# [Pipeline 3.4] ROC curves của tất cả tuned models
X_test_tune = feature_sets['full_features']['X_test']

cl.plot_roc_curves(
    tuned_models, X_test_tune, y_test_model,
    title='ROC Curves of Tuned Models'
)

<!-- @format -->

## [Pipeline 3.5] Final Comparison and Model Selection

Sau khi hoàn thành baseline training và hyperparameter tuning, nhóm tiến hành so sánh mô hình baseline tốt nhất với mô hình sau tuning tốt nhất.

Mục tiêu của bước này là chọn ra mô hình cuối cùng cho classical machine learning pipeline dựa trên các chỉ số đánh giá chính, đặc biệt là F1-score và ROC-AUC.


In [ ]:
# [Pipeline 3.5] So sánh best baseline vs best tuned model
final_comparison = cl.build_final_comparison(best_baseline, best_tuned)
final_comparison

In [ ]:
# [Pipeline 3.5] Tính mức cải thiện sau tuning
improvement_df = cl.compute_improvement(best_baseline, best_tuned)

In [ ]:
# [Pipeline 3.5] Biểu đồ so sánh các chỉ số
cl.plot_metric_comparison(final_comparison)

In [ ]:
# [Pipeline 3.5] Lưu bảng so sánh cuối
cl.save_results(final_comparison, 'final_model_comparison.csv', output_dir='../results')

<!-- @format -->

### Nhận xét kết quả Hyperparameter Tuning

Trong bước tinh chỉnh tham số, nhóm sử dụng `full_features` làm nhánh đặc trưng chính. Việc tuning được thực hiện bằng cross-validation trên tập train; tập test chỉ được dùng cho đánh giá cuối cùng để tránh data leakage.

Cross-validation được nâng từ 3 folds lên 5 folds để đánh giá ổn định hơn. Đối với Random Forest, `RandomizedSearchCV` với `n_iter=15` mở rộng phạm vi tìm kiếm nhưng vẫn giữ thời gian chạy phù hợp với môi trường Colab.

Mô hình cuối cùng được chọn dựa trên F1-score trên tập test, đồng thời tham khảo accuracy và ROC-AUC. Nếu mức cải thiện nhỏ, điều đó cho thấy cấu hình baseline ban đầu đã tương đối phù hợp với dữ liệu.


<!-- @format -->

### Final Model Selection

Dựa trên kết quả thực nghiệm, nhóm chọn mô hình có F1-score cao nhất trên tập test sau bước hyperparameter tuning làm mô hình cuối cùng.

Random Forest sau tuning được ưu tiên xem xét vì:

1. **F1-score là chỉ số chính** — cân bằng precision và recall, phù hợp với dữ liệu mất cân bằng lớp.
2. **ROC-AUC** — đánh giá khả năng tách biệt hai lớp trên nhiều ngưỡng quyết định.
3. **Phù hợp với dữ liệu tabular** — học quan hệ phi tuyến và tương tác giữa đặc trưng.
4. **Hỗ trợ feature importance** — giúp phân tích đặc trưng quan trọng.
5. **Không cần PCA** — `full_features` hoạt động tốt hơn `pca_95` với các mô hình chính.


<!-- @format -->

## [Pipeline 3.6] Feature Importance Analysis

Vì Random Forest là mô hình được ưu tiên lựa chọn sau tuning, nhóm phân tích thêm độ quan trọng của các đặc trưng. Phân tích này giúp kết nối kết quả mô hình với EDA và trả lời câu hỏi: những đặc trưng nào có ảnh hưởng lớn hơn đến dự đoán thu nhập.


In [ ]:
# [Pipeline 3.6] Phân tích feature importance của Random Forest
fi_df = cl.plot_feature_importance(
    tuned_models,
    best_preprocessor,
    num_features,
    cat_features,
    top_n=15,
    model_key='Random Forest'
)

<!-- @format -->

### Nhận xét Feature Importance

Biểu đồ feature importance cho thấy các đặc trưng có đóng góp lớn nhất vào quyết định phân loại của Random Forest. Với bài toán thu nhập trên Adult dataset, các đặc trưng liên quan đến trình độ học vấn, tình trạng hôn nhân, độ tuổi, số giờ làm việc mỗi tuần và các nhóm nghề nghiệp thường có ảnh hưởng đáng kể đến dự đoán.

Kết quả này phù hợp với phân tích EDA trước đó. Tuy nhiên, feature importance của Random Forest chỉ phản ánh mức độ đóng góp trong mô hình, không nên diễn giải như quan hệ nhân quả tuyệt đối.


In [ ]:
# [Pipeline 3.6] Lưu mô hình cuối cùng
import joblib

final_model_name = best_tuned['model']
final_model      = tuned_models[final_model_name]

model_path = cl.save_model(final_model, final_model_name, output_dir='../models')

# Verify load
loaded = cl.load_model(model_path)
from sklearn.metrics import f1_score, accuracy_score
X_test_tune = feature_sets['full_features']['X_test']
loaded_pred = loaded.predict(X_test_tune)
print('Loaded model F1-score:', round(f1_score(y_test_model, loaded_pred), 4))
print('Loaded model Accuracy:', round(accuracy_score(y_test_model, loaded_pred), 4))

<!-- @format -->

## Conclusion

Trong notebook này, nhóm đã xây dựng đầy đủ classical machine learning pipeline cho bài toán phân loại thu nhập trên Adult dataset.

Pipeline bao gồm các bước chính:

1. Tái lập preprocessing pipeline từ notebook 02.
2. Trích xuất đặc trưng với hai nhánh: `full_features` và `pca_95`.
3. Huấn luyện baseline với Logistic Regression, Linear SVM, Random Forest và Gaussian Naive Bayes.
4. Đánh giá bằng Accuracy, Precision, Recall, F1-score, ROC-AUC, Confusion Matrix và ROC Curve.
5. Tinh chỉnh siêu tham số bằng cross-validation (5-fold Stratified KFold).
6. So sánh kết quả baseline và tuned models.
7. Phân tích feature importance cho Random Forest.
8. Lưu features, kết quả và mô hình cuối cùng.

Nhánh `full_features` được chọn làm hướng chính vì PCA không cải thiện rõ rệt hiệu năng. Mô hình cuối cùng được chọn dựa trên F1-score trên tập test.
